# Evaluate a fixed FCR plan in one common environment

This source-only walkthrough asks a narrow research question: **given an already-authored hourly energy schedule and four-hour symmetric-FCR schedule, what does the common fixed-plan evaluator report for physical delivery, settlement, validation, and reproducibility?** It evaluates bids; it does not create or optimize them.

The default path is fully offline. It uses a small project-authored case for executable physics and accounting, then inspects the committed Golden Reference Fixture for bounded evidence about the pinned 2023-06-18 result. The source notebook is committed with empty outputs.

## 1. Terminology and evidence limits

Open Energy Lab uses **charge-positive** battery power. Positive activation requests upward discharge, so requested FCR power is `-activation × committed capacity`. Requested power is the command before physical limits; delivered power is the battery-terminal result after the shared battery transition. Stored energy is reported in kWh, while state of charge is the corresponding capacity fraction.

The report keeps directional FCR energy (upward discharge and downward charge), battery throughput, tracking, saturation, and settlement component reconciliation separately visible. A static invalid commitment returns `validation_failed` with no settlement or profit. Producer-neutral means that manual, heuristic, robust, and TPM labels do not select different physics or economics.

The Golden Reference Fixture supports **bounded reference-result parity** for declared aggregates. Published three-decimal decisions can yield `source_precision_compatible` rather than exact evidence. Missing detailed source evidence is `unavailable`; aggregate agreement is not used to infer it. There is no complete ten-second Julia trace parity claim.

## 2. Offline setup and explicit switches

From the repository root, install the optional notebook tools with `uv sync --group notebook`, then start this source notebook with `uv run jupyter lab examples/fcr_bid_evaluation/fcr_bid_evaluation_walkthrough.ipynb`. Select the project environment (the `.venv/bin/python` interpreter) as the kernel.

External reference replay and figure export are both disabled below. Enabling the reference branch never downloads or searches for data: it reads only the explicit caller-managed CSV path. Figure export, when selected, is restricted to the ignored temporary directory shown in the configuration.

In [ ]:
from pathlib import Path
import hashlib
import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from examples.fcr_bid_evaluation.run_manual_plan import build_manual_request
from oel_validation.reference_fcr import load_golden_reference_fixture
from open_energy_lab.simulator import evaluate_fixed_fcr_bid

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "pyproject.toml").is_file():
    raise RuntimeError("run this notebook from the Open Energy Lab repository root")

RUN_EXTERNAL_REFERENCE = False
EXPORT_FIGURES = False
EXTERNAL_REFERENCE_CSV = Path("artifacts/reference/robust_storage_bidding/regulation_signal.csv")
FIGURE_EXPORT_DIR = Path("tmp/fcr-bid-evaluation-notebook/figures")
GOLDEN_FIXTURE_PATH = Path("tests/fixtures/fcr_bid_evaluation/robust_storage_bidding_2023-06-18_aggregate_v1.json")

sns.set_theme(style="whitegrid", context="notebook")
generated_figures = {}

def canonical_fingerprint(value):
    payload = json.dumps(
        value, allow_nan=False, ensure_ascii=False, separators=(",", ":"), sort_keys=True
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

## 3. Author one fixed plan and evaluate it

The authored case reuses the same request builder as the command-line example. Its four hourly baseline decisions, one four-hour FCR commitment, eight half-hour activation intervals, battery, prices, and producer metadata are deterministic. Activation is evaluation input, not bidder-visible plan lineage.

In [ ]:
authored_request = build_manual_request()
authored_report = evaluate_fixed_fcr_bid(authored_request)
assert authored_report.status == "settled"
authored_payload = authored_report.to_dict()

evaluation_summary_table = pd.DataFrame(
    [
        {"item": "evaluation status", "value": authored_report.status},
        {"item": "producer kind", "value": authored_request.plan.producer.producer_kind},
        {"item": "objective", "value": authored_request.plan.objective.unavailable_reason},
        {"item": "objective bound", "value": authored_request.plan.objective_bound.unavailable_reason},
        {"item": "total profit (EUR)", "value": authored_report.total_profit_eur},
        {"item": "plan fingerprint", "value": authored_report.plan_fingerprint},
        {"item": "physical fingerprint", "value": authored_report.trace_fingerprint},
        {"item": "settlement fingerprint", "value": authored_report.settlement_fingerprint},
    ]
)
display(evaluation_summary_table)

In [ ]:
energy_schedule_payload = authored_request.energy_schedule.to_dict()
fcr_schedule_payload = authored_request.fcr_schedule.to_dict()
schedule_rows = [
    {
        "schedule": "energy baseline",
        "interval": index,
        "start_utc": item["interval"]["start_utc"],
        "end_utc": item["interval"]["end_utc"],
        "value_kw": item["power_kw"],
        "meaning": "charge-positive power",
    }
    for index, item in enumerate(energy_schedule_payload["intervals"])
]
schedule_rows.extend(
    {
        "schedule": "FCR capacity",
        "interval": index,
        "start_utc": item["interval"]["start_utc"],
        "end_utc": item["interval"]["end_utc"],
        "value_kw": item["capacity_kw"],
        "meaning": "symmetric reserve capacity",
    }
    for index, item in enumerate(fcr_schedule_payload["intervals"])
)
schedule_table = pd.DataFrame(schedule_rows)
activation_table = pd.DataFrame(
    [
        {
            "sample": index,
            "start_utc": item.interval.start_utc.isoformat(),
            "activation": item.activation,
        }
        for index, item in enumerate(authored_request.activation_trace.samples)
    ]
)
display(schedule_table, activation_table)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=False)
energy_rows = schedule_table[schedule_table["schedule"] == "energy baseline"]
fcr_rows = schedule_table[schedule_table["schedule"] == "FCR capacity"]
axes[0].step(energy_rows["interval"], energy_rows["value_kw"], where="mid", marker="o")
axes[0].axhline(0.0, color="black", linewidth=0.8)
axes[0].set(title="Authored hourly energy baseline", ylabel="Power (kW, charge-positive)")
axes[1].bar(fcr_rows["interval"], fcr_rows["value_kw"], color="#4c72b0", width=0.6)
axes[1].set(title="Authored four-hour symmetric FCR capacity", xlabel="Decision interval", ylabel="Capacity (kW)")
fig.tight_layout()
generated_figures["energy_and_fcr_schedules"] = fig
display(fig)

## 4. Inspect the common physical replay

Every row below comes from the evaluator's shared battery kernel. Activation and capacity determine requested FCR power; the baseline is added to form requested combined power. Delivered values, stored energy, SOC, tracking error, and saturation causes are physical results rather than bidder metadata.

In [ ]:
trace_samples = authored_payload["physical_trace"]["samples"]
physical_trace_table = pd.DataFrame(
    [
        {
            "sample": index,
            "start_utc": item["interval"]["start_utc"],
            "activation": item["activation"],
            "baseline_power_kw": item["baseline_power_kw"],
            "requested_fcr_power_kw": item["requested_fcr_power_kw"],
            "delivered_fcr_power_kw": item["delivered_fcr_power_kw"],
            "requested_combined_power_kw": item["requested_combined_power_kw"],
            "delivered_combined_power_kw": item["delivered_combined_power_kw"],
            "stored_energy_before_kwh": item["stored_energy_before_kwh"],
            "stored_energy_after_kwh": item["stored_energy_after_kwh"],
            "soc_after": item["soc_after"],
            "absolute_tracking_error_kw": item["absolute_tracking_error_kw"],
            "saturation": ", ".join(item["saturation_causes"]) or "none",
        }
        for index, item in enumerate(trace_samples)
    ]
)
metrics = authored_payload["physical_metrics"]
physical_summary_rows = [
    ("stored energy initial", metrics["stored_energy_kwh"]["initial"], "kWh"),
    ("stored energy minimum", metrics["stored_energy_kwh"]["minimum"], "kWh"),
    ("stored energy maximum", metrics["stored_energy_kwh"]["maximum"], "kWh"),
    ("stored energy terminal", metrics["stored_energy_kwh"]["terminal"], "kWh"),
    ("SOC initial", metrics["soc_fraction"]["initial"], "fraction"),
    ("SOC minimum", metrics["soc_fraction"]["minimum"], "fraction"),
    ("SOC maximum", metrics["soc_fraction"]["maximum"], "fraction"),
    ("SOC terminal", metrics["soc_fraction"]["terminal"], "fraction"),
    ("requested power minimum", metrics["requested_power_kw"]["minimum"], "kW"),
    ("requested power maximum", metrics["requested_power_kw"]["maximum"], "kW"),
    ("delivered power minimum", metrics["delivered_power_kw"]["minimum"], "kW"),
    ("delivered power maximum", metrics["delivered_power_kw"]["maximum"], "kW"),
    ("charge energy", metrics["charge_energy_kwh"], "kWh"),
    ("discharge energy", metrics["discharge_energy_kwh"], "kWh"),
    ("throughput", metrics["throughput_kwh"], "kWh"),
    ("absolute tracking sum", metrics["absolute_tracking_error_kw_sum"], "kW-samples"),
    ("non-delivery energy", metrics["non_delivery_energy_kwh"], "kWh"),
    ("saturation samples", metrics["saturation_sample_count"], "count"),
]
physical_summary_table = pd.DataFrame(physical_summary_rows, columns=["metric", "value", "unit"])
display(physical_trace_table, physical_summary_table)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
axes[0].step(physical_trace_table["sample"], physical_trace_table["activation"], where="mid", color="#8172b3")
axes[0].axhline(0.0, color="black", linewidth=0.8)
axes[0].set(title="Authored regulation activation", ylabel="Activation")
axes[1].plot(physical_trace_table["sample"], physical_trace_table["requested_combined_power_kw"], marker="o", label="requested")
axes[1].plot(physical_trace_table["sample"], physical_trace_table["delivered_combined_power_kw"], marker="s", linestyle="--", label="delivered")
axes[1].axhline(0.0, color="black", linewidth=0.8)
axes[1].set(title="Requested and delivered combined battery power", xlabel="Activation sample", ylabel="Power (kW)")
axes[1].legend()
fig.tight_layout()
generated_figures["activation_requested_delivered_power"] = fig
display(fig)

In [ ]:
state_checkpoint_table = pd.DataFrame(authored_payload["physical_trace"]["state_checkpoints"])
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
axes[0].plot(state_checkpoint_table["sample_index"], state_checkpoint_table["stored_energy_kwh"], marker="o", color="#55a868")
axes[0].axhline(authored_request.battery.capacity_kwh * authored_request.battery.min_soc, color="#c44e52", linestyle=":", label="minimum")
axes[0].axhline(authored_request.battery.capacity_kwh * authored_request.battery.max_soc, color="#c44e52", linestyle=":", label="maximum")
axes[0].set(title="Stored battery energy", ylabel="Energy (kWh)")
axes[0].legend()
axes[1].plot(state_checkpoint_table["sample_index"], state_checkpoint_table["soc"], marker="s", color="#4c72b0")
axes[1].axhline(authored_request.battery.min_soc, color="#c44e52", linestyle=":")
axes[1].axhline(authored_request.battery.max_soc, color="#c44e52", linestyle=":")
axes[1].set(title="Post-step state of charge", xlabel="Checkpoint / sample boundary", ylabel="SOC fraction")
fig.tight_layout()
generated_figures["stored_energy_and_soc"] = fig
display(fig)

## 5. Audit settlement, directional energy, and reconciliation

Available components carry numeric values; unavailable intraday, imbalance, degradation, and terminal-value models retain explicit reasons rather than numeric zero placeholders. Activation energy prices delivered FCR energy. Directional FCR energy and battery-terminal throughput remain distinct, and the displayed component reconciliation must close within the evaluator tolerance.

In [ ]:
settlement_payload = authored_payload["settlement"]
component_names = (
    "day_ahead_revenue", "fcr_capacity_revenue", "activation_energy_revenue",
    "penalty", "intraday", "imbalance", "degradation", "terminal_value",
)
settlement_table = pd.DataFrame(
    [
        {
            "component": name,
            "status": settlement_payload[name]["status"],
            "value_eur": settlement_payload[name]["value_eur"],
            "reason": settlement_payload[name]["reason"],
        }
        for name in component_names
    ]
)
directional_energy_rows = []
for basis in ("requested_fcr_energy_kwh", "delivered_fcr_energy_kwh"):
    for direction in ("upward_discharge", "downward_charge", "signed_net_charge_positive"):
        directional_energy_rows.append(
            {"basis": basis, "direction": direction, "value_kwh": metrics[basis][direction]}
        )
directional_energy_table = pd.DataFrame(directional_energy_rows)
reconciliation_table = pd.DataFrame(
    [
        {"quantity": "day-ahead revenue", "value_eur": settlement_payload["day_ahead_revenue"]["value_eur"]},
        {"quantity": "FCR capacity revenue", "value_eur": settlement_payload["fcr_capacity_revenue"]["value_eur"]},
        {"quantity": "delivered activation revenue", "value_eur": settlement_payload["activation_energy_revenue"]["value_eur"]},
        {"quantity": "penalty (subtracted)", "value_eur": -settlement_payload["penalty"]["value_eur"]},
        {"quantity": "total profit", "value_eur": settlement_payload["total_profit_eur"]},
        {"quantity": "absolute reconciliation error", "value_eur": settlement_payload["reconciliation_error_eur"]},
    ]
)
display(settlement_table, directional_energy_table, reconciliation_table)

In [ ]:
settlement_plot_table = settlement_table.copy()
settlement_plot_table["plot_value_eur"] = settlement_plot_table["value_eur"].fillna(0.0)
fig, axis = plt.subplots(figsize=(10, 5))
colors = ["#55a868" if status == "available" else "#c7c7c7" for status in settlement_plot_table["status"]]
axis.bar(settlement_plot_table["component"], settlement_plot_table["plot_value_eur"], color=colors)
axis.axhline(0.0, color="black", linewidth=0.8)
axis.set(title="Settlement components (grey means unavailable, not zero)", ylabel="EUR")
axis.tick_params(axis="x", rotation=35)
fig.tight_layout()
generated_figures["settlement_components"] = fig
display(fig)

## 6. Interpret a static invalid-headroom failure

The next plan commits more symmetric reserve than its baseline and battery power limits can support in both directions. This is a static accepted-commitment failure: the evaluator rejects it before ordinary physical replay and settlement. Diagnostics identify the failed field and evidence; no numeric profit is manufactured. A separately requested diagnostic replay may investigate an invalid commitment, but it is not valid bidder performance and is not parity-comparable.

In [ ]:
invalid_request = build_manual_request(invalid_headroom=True)
invalid_report = evaluate_fixed_fcr_bid(invalid_request)
assert invalid_report.status == "validation_failed"
invalid_payload = invalid_report.to_dict()
assert invalid_payload["physical_trace"] is None
assert invalid_payload["settlement"] is None
assert invalid_payload["total_profit_eur"] is None
diagnostic_table = pd.DataFrame(invalid_payload["diagnostics"])
diagnostic_interpretation_table = pd.DataFrame(
    [
        {"question": "Was ordinary replay accepted?", "answer": "No — static preflight failed"},
        {"question": "Is settlement available?", "answer": "No — invalid commitments have no profit"},
        {"question": "What should be changed?", "answer": "Reduce baseline/reserve commitment or change battery power limits"},
        {"question": "Can a diagnostic replay count as parity?", "answer": "No — it is explicitly non-economic investigation"},
    ]
)
display(diagnostic_table, diagnostic_interpretation_table)

## 7. Demonstrate producer-neutral evaluation

These four plans deliberately contain identical schedules, activation, battery, prices, and settlement inputs. Only the producer label changes. A TPM-labelled row demonstrates the evaluator boundary; it does **not** claim that TPM bid selection or bid generation exists. Physical and settlement fingerprints must remain identical even though plan and result lineage records the producer identity.

In [ ]:
producer_rows = []
producer_reports = {}
for producer_kind in ("manual", "heuristic", "robust", "tpm"):
    producer_request = build_manual_request(producer_kind=producer_kind)
    producer_report = evaluate_fixed_fcr_bid(producer_request)
    producer_reports[producer_kind] = producer_report
    producer_rows.append(
        {
            "producer_kind": producer_kind,
            "status": producer_report.status,
            "total_profit_eur": producer_report.total_profit_eur,
            "physical_fingerprint": producer_report.trace_fingerprint,
            "settlement_fingerprint": producer_report.settlement_fingerprint,
            "plan_fingerprint": producer_report.plan_fingerprint,
            "result_fingerprint": producer_report.result_fingerprint,
        }
    )
producer_comparison_table = pd.DataFrame(producer_rows)
assert producer_comparison_table["physical_fingerprint"].nunique() == 1
assert producer_comparison_table["settlement_fingerprint"].nunique() == 1
display(producer_comparison_table)

## 8. Inspect the offline Golden Reference Fixture

The committed fixture is a small provenance-rich evidence manifest, not the external regulation dataset and not a copied executable reference environment. It records the pinned generated result, mapping rules, 24 hourly energy decisions, six four-hour FCR decisions, price and battery inputs, 96 planning checkpoints, an external descriptor for 8,640 ten-second samples, supported fields, unavailable fields, and field-specific source-precision bounds. Loading it performs no network call and does not resolve its provenance-only source paths.

In [ ]:
golden_fixture = load_golden_reference_fixture(REPO_ROOT / GOLDEN_FIXTURE_PATH)
golden_payload = golden_fixture.to_dict()
assert len(golden_fixture.rounded_x0_kw) == 24
assert len(golden_fixture.rounded_xr_kw) == 6
assert golden_fixture.reference_configuration["planning_interval_count"] == 96
assert golden_fixture.external_activation.selected_row_count == 8640

In [ ]:
provenance = golden_fixture.reference_provenance
runtime = golden_fixture.runtime_provenance
reference_provenance_rows = [
    {"field": "fixture ID", "value": golden_fixture.fixture_id},
    {"field": "fixture fingerprint", "value": golden_fixture.fixture_fingerprint},
    {"field": "reference commit", "value": provenance["reference_commit"]},
    {"field": "generation command", "value": provenance["generation_command"]},
    {"field": "generation working directory", "value": provenance["generation_working_directory"]},
    {"field": "original result path", "value": provenance["original_result_path"]},
    {"field": "original result digest", "value": provenance["original_result_sha256"]},
    {"field": "recorded generation runtime", "value": runtime["generation_runtime"]["status"]},
    {"field": "external retention", "value": golden_fixture.external_activation.retention_status},
    {"field": "external content digest", "value": golden_fixture.external_activation.sha256},
]
reference_provenance_table = pd.DataFrame(reference_provenance_rows)
mapping = golden_fixture.mapping
reference_mapping_rows = [
    {"declaration": "mapping ID", "value": mapping.mapping_id},
    {"declaration": "source sign", "value": mapping.source_sign_convention},
    {"declaration": "target sign", "value": mapping.target_sign_convention},
    {"declaration": "x0 mapping", "value": mapping.x0_formula},
    {"declaration": "xr mapping", "value": mapping.xr_formula},
    {"declaration": "activation mapping", "value": mapping.activation_formula},
    {"declaration": "timezone interpretation", "value": mapping.source_timezone_interpretation},
    {"declaration": "target UTC start", "value": mapping.target_period.start_utc.isoformat()},
    {"declaration": "target UTC end", "value": mapping.target_period.end_utc.isoformat()},
    {"declaration": "index grids", "value": " | ".join(mapping.index_conversions)},
]
reference_mapping_table = pd.DataFrame(reference_mapping_rows)
reference_field_rows = [
    {
        "field_id": item.field_id,
        "evidence_status": ("supported_" + item.comparison_rule.replace("-", "_")),
        "source_path": item.reference_field_path,
        "unit": item.unit,
        "precision_bound_id": item.precision_bound_id,
    }
    for item in golden_fixture.expected_fields
]
reference_field_rows.extend(
    {
        "field_id": item.field_id,
        "evidence_status": "unavailable",
        "source_path": None,
        "unit": None,
        "precision_bound_id": None,
    }
    for item in golden_fixture.unsupported_fields
)
reference_field_table = pd.DataFrame(reference_field_rows)
precision_bound_rows = [
    {
        "bound_id": bound_id,
        "decision_bound": bound.decision_bound,
        "unit": bound.unit,
        "formula_id": bound.formula_id,
    }
    for bound_id, bound in golden_fixture.precision_policy.field_bounds.items()
]
precision_bound_table = pd.DataFrame(precision_bound_rows)
display(reference_provenance_table, reference_mapping_table, reference_field_table, precision_bound_table)

In [ ]:
reference_status_counts = (
    reference_field_table.groupby("evidence_status", sort=True).size().rename("field_count").reset_index()
)
fig, axes = plt.subplots(2, 1, figsize=(11, 9))
sns.barplot(data=reference_status_counts, x="evidence_status", y="field_count", ax=axes[0], color="#4c72b0")
axes[0].set(title="Golden Reference Fixture field evidence statuses", xlabel="Status", ylabel="Field count")
axes[0].tick_params(axis="x", rotation=25)
sns.barplot(data=precision_bound_table, x="bound_id", y="decision_bound", ax=axes[1], color="#dd8452")
axes[1].set(title="Analytic source-decision precision bounds (field units)", xlabel="Bound family", ylabel="Decision-only bound")
axes[1].tick_params(axis="x", rotation=30)
fig.tight_layout()
generated_figures["reference_statuses_and_precision_bounds"] = fig
display(fig)

In [ ]:
notebook_display_payload = {
    "schema_version": "oel.fcr-bid-evaluation-notebook-display.v1",
    "authored": {
        "status": authored_report.status,
        "plan_fingerprint": authored_report.plan_fingerprint,
        "physical_fingerprint": authored_report.trace_fingerprint,
        "settlement_fingerprint": authored_report.settlement_fingerprint,
        "result_fingerprint": authored_report.result_fingerprint,
        "physical_metrics": authored_payload["physical_metrics"],
        "settlement": authored_payload["settlement"],
    },
    "invalid": {
        "status": invalid_report.status,
        "diagnostics": invalid_payload["diagnostics"],
        "result_fingerprint": invalid_report.result_fingerprint,
    },
    "producer_rows": producer_rows,
    "reference": {
        "fixture_fingerprint": golden_fixture.fixture_fingerprint,
        "field_rows": reference_field_rows,
        "precision_bound_rows": precision_bound_rows,
    },
    "figure_keys": sorted(generated_figures),
}
notebook_display_fingerprint = canonical_fingerprint(notebook_display_payload)
display(pd.DataFrame([{"notebook_display_fingerprint": notebook_display_fingerprint}]))

## 9. Optional explicit-path 2023-06-18 replay

This branch is deliberately disabled. To run it, first install and digest-check the external CSV exactly as documented in the Feature 012 quickstart, set `RUN_EXTERNAL_REFERENCE = True`, and set `EXTERNAL_REFERENCE_CSV` to that explicit path. The existing reference request builder validates the complete file, selects the declared day, maps 24/6/96/8,640 intervals, runs the common evaluator, and compares only supported aggregates. It never downloads, searches for, or infers the CSV. When selected, the unit-separated parity figure after the table shows Julia expected values and Open Energy Lab actual values on the same categorical row, together with their available source-precision intervals. A second column reports the signed deviation divided by the allowed precision envelope; values inside ±1 are source-precision compatible.

In [ ]:
external_reference_result = {"status": "not_requested"}
external_reference_comparison_table = pd.DataFrame()
if RUN_EXTERNAL_REFERENCE:
    from examples.fcr_bid_evaluation.run_reference_parity import build_reference_request
    from open_energy_lab.simulator import compare_reference_evaluation

    external_request = build_reference_request(
        fixture_path=REPO_ROOT / GOLDEN_FIXTURE_PATH,
        activation_path=EXTERNAL_REFERENCE_CSV.resolve(),
    )
    external_report = evaluate_fixed_fcr_bid(external_request)
    if external_report.status != "settled":
        raise RuntimeError(f"external reference evaluation did not settle: {external_report.status}")
    external_comparison = compare_reference_evaluation(external_request, external_report)
    external_comparison_rows = [item.to_dict() for item in external_comparison.fields]
    external_reference_comparison_table = pd.DataFrame(external_comparison_rows)
    external_reference_result = {
        "status": external_comparison.overall_status,
        "counts": {
            "energy": len(external_request.energy_schedule.intervals),
            "fcr": len(external_request.fcr_schedule.intervals),
            "checkpoint": len(external_request.time_contract.index.checkpoint_intervals),
            "activation": len(external_request.activation_trace.samples),
        },
        "mismatch_count": sum(
            item.status == "mismatch"
            for item in external_comparison.fields
        ),
        "trace_fingerprint": external_report.trace_fingerprint,
        "settlement_fingerprint": external_report.settlement_fingerprint,
        "comparison_fingerprint": external_comparison.comparison_fingerprint,
        "result_fingerprint": external_report.result_fingerprint,
    }
display(pd.DataFrame([external_reference_result]), external_reference_comparison_table)

In [ ]:
external_reference_parity_plot_table = pd.DataFrame()
external_reference_parity_figure = None
if RUN_EXTERNAL_REFERENCE:
    from matplotlib.lines import Line2D

    parity_units = ("EUR", "kWh", "kW")
    external_reference_parity_plot_table = external_reference_comparison_table.loc[
        external_reference_comparison_table["unit"].isin(parity_units)
        & external_reference_comparison_table["actual_value"].notna()
        & external_reference_comparison_table["expected_value"].notna()
    ].copy()
    external_reference_parity_plot_table["signed_deviation"] = (
        external_reference_parity_plot_table["actual_value"]
        - external_reference_parity_plot_table["expected_value"]
    )
    external_reference_parity_plot_table["allowed_precision_envelope"] = (
        external_reference_parity_plot_table["decision_bound"].fillna(0.0)
        + external_reference_parity_plot_table["aggregate_half_unit"].fillna(0.0)
    )
    external_reference_parity_plot_table["normalized_deviation"] = [
        deviation / allowance
        if allowance > 0.0
        else (0.0 if abs(deviation) <= 1e-12 else None)
        for deviation, allowance in zip(
            external_reference_parity_plot_table["signed_deviation"],
            external_reference_parity_plot_table["allowed_precision_envelope"],
            strict=True,
        )
    ]
    status_colors = {
        "exact": "#55a868",
        "source_precision_compatible": "#dd8452",
        "mismatch": "#c44e52",
        "unavailable": "#8c8c8c",
    }
    external_reference_parity_figure, axes = plt.subplots(
        len(parity_units), 2, figsize=(17, 13), constrained_layout=True, squeeze=False
    )
    for unit_index, unit in enumerate(parity_units):
        value_axis = axes[unit_index, 0]
        deviation_axis = axes[unit_index, 1]
        unit_table = (
            external_reference_parity_plot_table.loc[
                external_reference_parity_plot_table["unit"] == unit
            ]
            .sort_values("field_id")
            .reset_index(drop=True)
        )
        for row_index, row in unit_table.iterrows():
            color = status_colors.get(row["status"], "#8c8c8c")
            categorical_y = row_index
            if isinstance(row["expected_interval"], (list, tuple)):
                value_axis.hlines(categorical_y, *row["expected_interval"], color=color, linewidth=8, alpha=0.22)
            if isinstance(row["actual_interval"], (list, tuple)):
                value_axis.hlines(categorical_y, *row["actual_interval"], color=color, linewidth=3, alpha=0.9)
            value_axis.plot(
                [row["expected_value"], row["actual_value"]],
                [categorical_y, categorical_y],
                color=color, linewidth=1, alpha=0.65,
            )
            value_axis.scatter(row["expected_value"], categorical_y, marker="o", facecolors="white", edgecolors=color, s=70, zorder=3)
            value_axis.scatter(row["actual_value"], categorical_y, marker="x", color=color, s=70, linewidths=2, zorder=4)
        row_colors = [status_colors.get(status, "#8c8c8c") for status in unit_table["status"]]
        deviation_axis.axvspan(-1.0, 1.0, color="#55a868", alpha=0.08)
        deviation_axis.axvline(0.0, color="black", linewidth=0.8)
        deviation_axis.axvline(-1.0, color="#55a868", linestyle=":", linewidth=1.2)
        deviation_axis.axvline(1.0, color="#55a868", linestyle=":", linewidth=1.2)
        deviation_axis.barh(
            range(len(unit_table)), unit_table["normalized_deviation"],
            color=row_colors, height=0.48, alpha=0.85,
        )
        finite_deviations = [
            abs(float(value))
            for value in unit_table["normalized_deviation"]
            if pd.notna(value)
        ]
        deviation_extent = max([1.15, *(value * 1.25 for value in finite_deviations)])
        deviation_axis.set_xlim(-deviation_extent, deviation_extent)
        for row_index, value in enumerate(unit_table["normalized_deviation"]):
            if pd.notna(value):
                deviation_axis.text(
                    float(value), row_index, f" {float(value):+.2f}×",
                    va="center", ha=("left" if value >= 0.0 else "right"), fontsize=8,
                )
        value_axis.set_yticks(range(len(unit_table)), unit_table["field_id"])
        value_axis.set(title=f"Julia expected vs OEL actual — {unit}", xlabel=unit, ylabel="Reference field (categorical)")
        value_axis.grid(axis="x", alpha=0.3)
        deviation_axis.set_yticks(range(len(unit_table)), [])
        deviation_axis.set(title="Deviation relative to precision allowance", xlabel="(OEL actual − Julia expected) / allowed envelope")
        deviation_axis.grid(axis="x", alpha=0.3)
    legend_handles = [
        Line2D([0], [0], marker="o", markerfacecolor="white", markeredgecolor="black", linestyle="None", label="Julia expected"),
        Line2D([0], [0], marker="x", color="black", linestyle="None", label="OEL actual"),
    ]
    legend_handles.extend(
        Line2D([0], [0], color=color, linewidth=4, label=status.replace("_", " "))
        for status, color in status_colors.items()
        if status in set(external_reference_parity_plot_table["status"])
    )
    axes[0, 0].legend(handles=legend_handles, loc="best", fontsize=8)
    generated_figures["external_reference_parity"] = external_reference_parity_figure
    display(external_reference_parity_figure)

In [ ]:
exported_figure_paths = []
if EXPORT_FIGURES:
    export_directory = REPO_ROOT / FIGURE_EXPORT_DIR
    export_directory.mkdir(parents=True, exist_ok=True)
    for figure_name, figure in sorted(generated_figures.items()):
        destination = export_directory / f"{figure_name}.png"
        figure.savefig(destination, dpi=150, bbox_inches="tight", metadata={"Software": "Open Energy Lab"})
        exported_figure_paths.append(destination)
display(pd.DataFrame({"exported_figure": [str(path) for path in exported_figure_paths]}))

## 10. Conclusions and remaining scope

This walkthrough establishes that one common fixed-plan evaluator can replay an authored energy-plus-FCR schedule, expose requested and delivered physics, reconcile available settlement components, reject static invalid headroom without profit, and preserve identical physical and settlement results across producer labels. It also makes the Golden Reference Fixture's provenance, 24/6 mapping, supported/unavailable fields, and field-specific precision bounds inspectable offline.

The optional complete-day branch can strengthen supported aggregate evidence when the caller explicitly supplies the governed CSV. It still does not establish bid optimization, uncertainty-set construction, TPM bid selection, or complete Julia trace parity. Those remain out of scope; a TPM-labelled fixed plan here demonstrates evaluator neutrality only.